## ระบบลงทะเบียนคอร์สเรียนโรงเรียนกวดวิชา
Coding Challenge — หัวข้อที่ 11
**องค์กร**: โรงเรียนกวดวิชา **ระบบ**: ระบบลงทะเบียนคอร์สเรียน

ระบบจำลองนี้ออกแบบตามรายละเอียดของกลุ่ม โดยมีนักเรียน (Student) เป็นผู้เริ่มต้นกระบวนการ นักเรียนเลือกคอร์ส (Course) จากนั้นระบบตรวจสอบจำนวนที่นั่ง หากยังมีที่ว่างจะสร้างรายการลงทะเบียน (Enrollment) คำนวณค่าเรียนรวม และบันทึกการชำระเงิน (Payment)

ข้อมูลหลักของระบบแบ่งเป็น 4 ส่วน ได้แก่ Student, Course, Enrollment และ Payment โดย Enrollment ทำหน้าที่เชื่อมข้อมูลระหว่าง Student และ Course


##ส่วนที่ 1 - ขั้นตอนการทำงานของระบบ
นักเรียน 1 คนสามารถเลือกได้ 1 คอร์สต่อการลงทะเบียน 1 ครั้ง โดยระบบมีขั้นตอนการทำงานดังนี้
**นักเรียนเลือกคอร์ส → ตรวจสอบที่นั่ง → ลงทะเบียน → คำนวณค่าเรียน → ชำระเงิน → จัดเก็บข้อมูล → วิเคราะห์ข้อมูล**

### ข้อมูลหลัก
- **Student:** รหัสนักเรียน ชื่อ เพศ อายุ ระดับชั้น โรงเรียน และเบอร์โทรศัพท์
- **Course:** รหัสคอร์ส ชื่อคอร์ส วิชา ครูผู้สอน ราคา จำนวนที่นั่ง และจำนวนผู้สมัคร
- **Enrollment:** รหัสการลงทะเบียน รหัสนักเรียน รหัสคอร์ส วันที่ลงทะเบียน ส่วนลด ค่าเรียนรวม และสถานะ
- **Payment:** รหัสการชำระเงิน รหัสการลงทะเบียน วันที่ชำระ จำนวนเงิน วิธีชำระ และสถานะ

ในงานนี้จะจำลองข้อมูลนักเรียน 450 คน และจำลองการลงทะเบียน 450 รายการ โดยใช้ for loop และ random เพื่อจำลองการทำงานของระบบทีละรายการ จากนั้นแปลงข้อมูลเป็น DataFrame และบันทึกเป็นไฟล์ CSV และฐานข้อมูล SQLite เพื่อนำไปวิเคราะห์ข้อมูลด้วย pandas และ SQL


## ส่วนที่ 2 — เตรียม class
ระบบประกอบด้วย 4 Class หลัก ได้แก่ `Student`, `Course`, `Enrollment` และ `Payment`

แต่ละ Class มี Attribute สำหรับเก็บข้อมูล และ Method สำหรับการทำงานของระบบจริง

In [ ]:
import random
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

random.seed(11)
print("เตรียม Library เรียบร้อยแล้ว")

เตรียม Library เรียบร้อยแล้ว


In [ ]:
class Student:
    def __init__(self, student_id, name, gender, age, grade, school, phone):
        self.student_id = student_id
        self.name = name
        self.gender = gender
        self.age = age
        self.grade = grade
        self.school = school
        self.phone = phone

    def get_grade(self):
        return self.grade

class Course:
    def __init__(self, course_id, course_name, subject, teacher, price, capacity):
        self.course_id = course_id
        self.course_name = course_name
        self.subject = subject
        self.teacher = teacher
        self.price = price
        self.capacity = capacity
        self.enrolled_count = 0

    def has_seat(self):
        return self.enrolled_count < self.capacity

    def add_student(self):
        if self.has_seat():
            self.enrolled_count += 1
            return True
        return False

    def available_seats(self):
        return self.capacity - self.enrolled_count


class Enrollment:
    def __init__(
        self,
        enrollment_id,
        student,
        course,
        enroll_date,
        discount_rate=0
    ):
        self.enrollment_id = enrollment_id
        self.student = student
        self.course = course
        self.enroll_date = enroll_date
        self.discount_rate = discount_rate
        self.total_tuition = 0
        self.status = "รอตรวจสอบ"

    def calculate_total(self):
        discount = self.course.price * self.discount_rate
        self.total_tuition = round(
            self.course.price - discount,
            2
        )
        return self.total_tuition

    def confirm(self):
        if self.course.add_student():
            self.status = "ลงทะเบียนสำเร็จ"
            self.calculate_total()
            return True

        self.status = "คอร์สเต็ม"
        return False


class Payment:
    def __init__(
        self,
        payment_id,
        enrollment,
        payment_date,
        method
    ):
        self.payment_id = payment_id
        self.enrollment = enrollment
        self.payment_date = payment_date
        self.amount = enrollment.total_tuition
        self.method = method
        self.status = "ชำระเงินแล้ว"

    def record_payment(self):
        return self.status


print("สร้าง Class ทั้ง 4 Class สำเร็จ")

สร้าง Class ทั้ง 4 Class สำเร็จ


## ส่วนที่ 3 —  เขียนฟังก์ชันช่วยงาน (Helper Functions)

In [ ]:

# เตรียมข้อมูลสำหรับสร้างนักเรียน

FIRST_NAMES = {
    "ชาย": [
        "กิตติ", "ธนภัทร", "ณัฐพงศ์", "พีรพัฒน์",
        "วรเมธ", "ภูริณัฐ", "ชยพล", "ภาคิน",
        "ณัฐวุฒิ", "ศุภกร", "ธนกร", "พศิน",
        "กวิน", "นราวิชญ์", "ปวริศ", "รชต",
        "ธีรภัทร", "ศุภณัฐ", "ภาณุวัฒน์", "พีรวิชญ์",
        "วชิรวิทย์", "ณรงค์ฤทธิ์", "เอกภพ", "สิรภพ",
        "กิตติพงษ์", "ชยุต", "ธนวัฒน์", "ภัทรพล",
        "ณัฐดนัย", "กรวิชญ์", "พีรพงศ์", "ศุภชัย",
        "ธีรเดช", "ปกรณ์", "ภัทรดนัย", "วรากร",
        "กฤติน", "ณัฐภัทร", "ภูวดล", "ธนาธิป",
        "รวิภาส", "ชนนท์", "ก้องภพ", "พงศกร",
        "สิทธิโชค", "อธิพัฒน์", "ปัณณวิชญ์", "ธีรภัทร"
    ],

    "หญิง": [
        "พิมพ์ชนก", "ณิชาภา", "กัญญาวีร์", "ศิริพร",
        "ชลธิชา", "ปภาวดี", "ธัญชนก", "วรัญญา",
        "ณัฐธิดา", "สุภัสสรา", "พิชญาภา", "กมลชนก",
        "ชนัญชิดา", "พัชราภา", "ศศิธร", "ปิยธิดา",
        "ณิชกานต์", "กุลธิดา", "ธนพร", "สิริกัญญา",
        "พิมพ์ลภัส", "อริสา", "ชญาดา", "วริศรา",
        "กัญญาณัฐ", "ณัฐณิชา", "สุพิชญา", "ภัทรวดี",
        "ธัญญารัตน์", "ปุณณภา", "กานต์พิชชา", "นภัสสร",
        "ศุภัคสร", "พิชญ์สินี", "รินรดา", "เขมิกา",
        "อัญชิสา", "ชุติกาญจน์", "พัณณิตา", "ญาดา",
        "ณิชาภัทร", "ปวีณา", "วรรณวิสา", "สุชาดา",
        "พัชรินทร์", "กุลจิรา", "ธิดารัตน์", "อริสรา"
    ]
}

LAST_NAMES = [
    "ใจดี", "ศรีสุข", "สายทอง", "วงศ์ดี", "แก้วมณี",
    "บุญมี", "แสงทอง", "พัฒนกุล", "สุวรรณ", "ธรรมรักษ์",
    "ศรีสวัสดิ์", "เจริญสุข", "วิไลวรรณ", "ทองดี", "บุญส่ง",
    "ศรีวงศ์", "สมบูรณ์", "รัตนวงศ์", "พงษ์ไพบูลย์", "ชัยมงคล",
    "วัฒนกุล", "กิตติชัย", "ศรีเจริญ", "บุญญฤทธิ์", "ไพศาล",
    "วงศ์สุวรรณ", "ธรรมวงศ์", "มงคลชัย", "พรหมรักษา", "จันทร์ดี",
    "ทองคำ", "ศรีบุญเรือง", "บุญเรือง", "วรรณศรี", "สุขเกษม",
    "รัตนชัย", "พูนทรัพย์", "เกียรติศักดิ์", "ชูศรี", "อินทร์แก้ว",
    "แสงจันทร์", "ทองสุข", "บุญธรรม", "ศรีทอง", "แก้วคำ",
    "วงศ์ษา", "สวัสดิ์ชัย", "เจริญวงศ์", "สุริยะ", "ไชยวงศ์",
    "พรมมา", "คำภา", "ศรีสมบัติ", "บุญเลิศ", "ทองใบ",
    "ศรีวิชัย", "สุขสันต์", "รัตนกุล", "วัฒนชัย", "ชนะชัย",
    "พงษ์ศรี", "จันทร์แก้ว", "บุญชู", "ศรีนวล", "แก้วใส",
    "ธรรมวงศ์", "สุวรรณดี", "เจริญทรัพย์", "มณีรัตน์", "ศรีอุดม"
]
SCHOOLS = [
    "โรงเรียนสาธิตมหาวิทยาลัยขอนแก่น ฝ่ายมัธยมศึกษา (ศึกษาศาสตร์)",
    "โรงเรียนสาธิตมหาวิทยาลัยขอนแก่น ฝ่ายมัธยมศึกษา (มอดินแดง)",
    "โรงเรียนขอนแก่นวิทยายน",
    "โรงเรียนกัลยาณวัตร",
    "โรงเรียนแก่นนครวิทยาลัย",
    "โรงเรียนขามแก่นนคร",
    "โรงเรียนนครขอนแก่น",
    "โรงเรียนขอนแก่นวิทยาลัย",
    "โรงเรียนเตรียมอุดมศึกษาพัฒนาการ ขอนแก่น",
    "โรงเรียนขอนแก่นพัฒนศึกษา"
]

GRADE = ["ม.4", "ม.5", "ม.6"]

PAYMENT_METHODS = [
    "โอนธนาคาร",
    "พร้อมเพย์",
    "บัตรเครดิต",
    "เงินสด"
]


In [ ]:
#เตรียมข้อมูลคอร์ส
COURSE_MASTER = [

    # ---------------- ม.ปลาย เนื้อหาพื้นฐาน ----------------

    {
        "course_id": "C001",
        "course_name": "คณิตศาสตร์ ม.ปลาย",
        "subject": "คณิตศาสตร์",
        "teacher": "ครูพิมพ์",
        "price": 2800,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    {
        "course_id": "C002",
        "course_name": "ฟิสิกส์ ม.ปลาย",
        "subject": "ฟิสิกส์",
        "teacher": "ครูเอก",
        "price": 2800,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    {
        "course_id": "C003",
        "course_name": "เคมี ม.ปลาย",
        "subject": "เคมี",
        "teacher": "ครูศิริ",
        "price": 2700,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    {
        "course_id": "C004",
        "course_name": "ชีววิทยา ม.ปลาย",
        "subject": "ชีววิทยา",
        "teacher": "ครูแพรว",
        "price": 2700,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    {
        "course_id": "C005",
        "course_name": "ภาษาอังกฤษ ม.ปลาย",
        "subject": "ภาษาอังกฤษ",
        "teacher": "ครูณัฐ",
        "price": 2500,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    # ---------------- ติวสอบ A-Level ----------------

    {
        "course_id": "C006",
        "course_name": "คณิตศาสตร์ A-Level",
        "subject": "คณิตศาสตร์",
        "teacher": "ครูพิมพ์",
        "price": 3500,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    {
        "course_id": "C007",
        "course_name": "ฟิสิกส์ A-Level",
        "subject": "ฟิสิกส์",
        "teacher": "ครูเอก",
        "price": 3500,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    {
        "course_id": "C008",
        "course_name": "เคมี A-Level",
        "subject": "เคมี",
        "teacher": "ครูศิริ",
        "price": 3400,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    {
        "course_id": "C009",
        "course_name": "ชีววิทยา A-Level",
        "subject": "ชีววิทยา",
        "teacher": "ครูแพรว",
        "price": 3300,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },

    {
        "course_id": "C010",
        "course_name": "ภาษาอังกฤษ A-Level",
        "subject": "ภาษาอังกฤษ",
        "teacher": "ครูณัฐ",
        "price": 3200,
        "max_seat": 40,
        "target_level": "ม.ปลาย"
    },]

In [ ]:
#สร้าง Course Object จาก COURSE_MASTER
courses = {}

for data in COURSE_MASTER:
  course = Course(data["course_id"], data["course_name"], data["subject"], data["teacher"], data["price"], data["max_seat"],)
  courses[course.course_id] = course

print("สร้างคอร์สสำเร็จ:", len(courses), "คอร์ส")


สร้างคอร์สสำเร็จ: 10 คอร์ส


In [ ]:
df = pd.DataFrame([
    {
        "course_id": c.course_id,
        "course_name": c.course_name,
        "subject": c.subject,
        "teacher": c.teacher,
        "price": c.price,
        "capacity": c.capacity,
        "enrolled_count": c.enrolled_count,
        "available_seats": c.available_seats()
    }
    for c in courses.values() ])

df.index = df.index + 1

display(df)

,course_id,course_name,subject,teacher,price,capacity,enrolled_count,available_seats
1,C001,คณิตศาสตร์ ม.ปลาย,คณิตศาสตร์,ครูพิมพ์,2800,40,0,40
2,C002,ฟิสิกส์ ม.ปลาย,ฟิสิกส์,ครูเอก,2800,40,0,40
3,C003,เคมี ม.ปลาย,เคมี,ครูศิริ,2700,40,0,40
4,C004,ชีววิทยา ม.ปลาย,ชีววิทยา,ครูแพรว,2700,40,0,40
5,C005,ภาษาอังกฤษ ม.ปลาย,ภาษาอังกฤษ,ครูณัฐ,2500,40,0,40
6,C006,คณิตศาสตร์ A-Level,คณิตศาสตร์,ครูพิมพ์,3500,40,0,40
7,C007,ฟิสิกส์ A-Level,ฟิสิกส์,ครูเอก,3500,40,0,40
8,C008,เคมี A-Level,เคมี,ครูศิริ,3400,40,0,40
9,C009,ชีววิทยา A-Level,ชีววิทยา,ครูแพรว,3300,40,0,40
10,C010,ภาษาอังกฤษ A-Level,ภาษาอังกฤษ,ครูณัฐ,3200,40,0,40
